In [ ]:
import os
from collections.abc import Sequence

import numpy as np
import sinter

from qldpc import circuits, codes
from qldpc.circuits import (
    DepolarizingNoiseModel,
    EdgeColoring,
    SinterDecoder,
    memory_experiment,
)
from qldpc.objects import Pauli, PauliXZ

### Toric Code Memory Experiments

In [ ]:
def run_experiments(basis: PauliXZ, distances: Sequence[int], error_rates: Sequence[float]):
    sm_compiler = EdgeColoring()
    tasks: list[sinter.Task] = []
    for distance in distances:
        toric_code = codes.ToricCode(distance, rotated=False)
        for prob in error_rates:
            noise_model = DepolarizingNoiseModel(prob, include_idling_error=False)
            tasks.append(
                sinter.Task(
                    circuit=memory_experiment(
                        toric_code, sm_compiler, distance, basis, noise_model
                    ),
                    json_metadata={"d": distance, "p": prob},
                )
            )

    return sinter.collect(
        num_workers=os.cpu_count() - 2,
        max_shots=10**6,
        max_errors=100,
        tasks=tasks,
        decoders=["bplsd"],
        custom_decoders={
            "bplsd": SinterDecoder(
                with_BP_LSD=True,
                max_iter=30,
                bp_method="ms",
                lsd_method="lsd_cs",
                lsd_order=0,
            )
        },
    )

In [ ]:
distances = [3, 5, 7]
error_rates = np.logspace(-3, -2, 5)
z_basis_results = run_experiments(Pauli.Z, distances, error_rates)
x_basis_results = run_experiments(Pauli.X, distances, error_rates)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, sharey=True, figsize=(8, 4))
sinter.plot_error_rate(
    ax=ax[0],
    stats=z_basis_results,
    x_func=lambda stats: stats.json_metadata["p"],
    group_func=lambda stats: stats.json_metadata["d"],
)
ax[0].loglog()
ax[0].grid(which="both")
ax[0].legend()
ax[0].set_ylabel("Logical Error Rate")
ax[0].set_xlabel("Uniform Physical Error Rate")
ax[0].set_title("Z Basis")

sinter.plot_error_rate(
    ax=ax[1],
    stats=x_basis_results,
    x_func=lambda stats: stats.json_metadata["p"],
    group_func=lambda stats: stats.json_metadata["d"],
)
ax[1].loglog()
ax[1].grid(which="both")
ax[1].legend()
ax[1].set_xlabel("Uniform Physical Error Rate")
ax[1].set_title("X Basis")

plt.suptitle("Toric Code Memory Experiments", fontsize=14)
plt.show()